# user

In [ ]:
VERSION = "0.0.1"
PSF_NAME = "psf_style-airy_size-41_pixelscale-0.1_sampling-1"
FILTER_NAME = "gauss_3.0_7x7.conv"
PIXEL_SCALE = 0.1

# lib

In [ ]:
import numpy
from pandas import read_csv, concat
from plots import (
    DIRS,
    case_result,
    case_result_param,
    ellips_model,
    extra_psf_images,
    extra_psf_images_reshaped,
    gauss_conv,
    hdul_object_ldac,
    histos,
    image_fitting,
    ldac,
    background,
    get_match,
    logged_values,
)
from tqdm import tqdm

DATASET_DIR_ORIGIN = DIRS["pisco_euclid"] / VERSION
DATASET_DIR = DIRS["pisco_euclid_se"] / VERSION
PSF_DIR = DIRS["RESULTS_SEXTRACTOR"] / PSF_NAME
RESULTS_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION
CATALOG_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "CATALOG"
IMAGES_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "IMAGES"
PLOTS_DIR = DIRS["RESULTS_SEXTRACTOR"] / VERSION / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# filter

In [ ]:
gauss_conv(DIRS["psf_se"] / FILTER_NAME, PLOTS_DIR)

# ldac

In [ ]:
objects_data = ldac(PSF_DIR, PSF_NAME)

In [ ]:
shape_vignet = hdul_object_ldac(objects_data, PLOTS_DIR)

# extra images psf

In [ ]:
extra_psf_images(DIRS["psf_se"], PSF_NAME, PSF_DIR, PLOTS_DIR, loggy=False)
extra_psf_images(DIRS["psf_se"], PSF_NAME, PSF_DIR, PLOTS_DIR, loggy=True)

In [ ]:
extra_psf_images_reshaped(PSF_DIR, PSF_NAME, shape_vignet, PLOTS_DIR, style="samples")
extra_psf_images_reshaped(PSF_DIR, PSF_NAME, shape_vignet, PLOTS_DIR, style="residuals")

# extra

In [ ]:
CHOICE = "000025_simu"

image_fitting(DATASET_DIR, IMAGES_DIR, CHOICE, PLOTS_DIR)
image_fitting(DATASET_DIR, IMAGES_DIR, CHOICE, PLOTS_DIR, transformed=True)

In [ ]:
background(IMAGES_DIR / f"back_{CHOICE}")

# inference

In [ ]:
# load parameters
true_images_params = read_csv(
    DATASET_DIR_ORIGIN / "images_parameters_true.csv", sep=",", skiprows=0
)
true_galaxies_params = read_csv(
    DATASET_DIR_ORIGIN / "galaxies_parameters_true.csv", sep=",", skiprows=0
)
PRED_FILES = sorted(list((CATALOG_DIR).glob("*_simu.csv")))
TRUE_FILES = sorted(list((DATASET_DIR).glob("*_true_galaxies_params.csv")))

# inference
pred_avg_e1, pred_avg_e2, detections, pred_galaxies_params, slice_pred, slice_truth = [
    [] for _ in range(6)
]
for i, (result_file, true_file) in enumerate(
    tqdm(zip(PRED_FILES, TRUE_FILES), total=len(PRED_FILES))
):
    # load results
    df_pred = read_csv(result_file).sort_values("X_IMAGE")
    df_pred = df_pred.dropna(subset=["ELLIP1ERRMODEL_IMAGE", "ELLIP2ERRMODEL_IMAGE"])
    detections.append(df_pred.shape[0])
    df_true = read_csv(true_file).sort_values("x")

    # slices
    temp_pred, temp_truth = get_match(df_pred, df_true, radius=2)
    temp_pred["image"] = i
    temp_truth["image"] = i
    slice_pred.append(temp_pred)
    slice_truth.append(temp_truth)

    # predictions galaxies
    df_pred["image"] = i
    pred_galaxies_params.append(df_pred)

    # predictions images
    avg_e1_temp, avg_e2_temp = ellips_model(
        df_pred["ELLIP1ERRMODEL_IMAGE"].to_numpy(),
        df_pred["ELLIP2ERRMODEL_IMAGE"].to_numpy(),
        df_pred["ELLIP1MODEL_IMAGE"].to_numpy(),
        df_pred["ELLIP2MODEL_IMAGE"].to_numpy(),
    )
    pred_avg_e1.append(avg_e1_temp)
    pred_avg_e2.append(avg_e2_temp)

# concatenations
pred_galaxies_params = concat(pred_galaxies_params, ignore_index=True)
slice_pred = concat(slice_pred, ignore_index=True)
slice_truth = concat(slice_truth, ignore_index=True)
detections = numpy.array(detections)
pred_avg_e1 = numpy.array(pred_avg_e1)
pred_avg_e2 = numpy.array(pred_avg_e2)

# histo

In [ ]:
bins = 100
logged_values(bins, pred_galaxies_params, true_galaxies_params, PIXEL_SCALE, PLOTS_DIR)

In [ ]:
bins = 50
density = True

histos(
    detections,
    pred_galaxies_params,
    true_galaxies_params,
    pred_avg_e1,
    pred_avg_e2,
    true_images_params,
    PLOTS_DIR,
    PIXEL_SCALE,
    bins=bins,
    density=density,
)

del bins, density

# results

In [ ]:
case_result(
    pred_avg_e1,
    pred_avg_e2,
    true_images_params["avg_g1"],
    true_images_params["avg_g2"],
    PLOTS_DIR / "results_se.pdf",
)

In [ ]:
case_result(
    true_images_params["avg_obs_e1"],
    true_images_params["avg_obs_e2"],
    true_images_params["avg_g1"],
    true_images_params["avg_g2"],
    PLOTS_DIR / "results_avg_e_g_best.pdf",
)